# Diabetes Prediction (S5E12) — Exploratory Data Analysis

An 8-question framework over the training data (700k rows of clinical, lifestyle,
and demographic features). Run from the project root after `data/raw/train.csv`
and `test.csv` are downloaded. Figures are saved to `reports/figures/`; copy
conclusions into `reports/EDA_FINDINGS.md`.

**Task:** binary classification · **Metric:** ROC-AUC.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
FIG = ROOT / 'reports' / 'figures'
FIG.mkdir(parents=True, exist_ok=True)

from src.data import (
    BINARY_FEATURES, NUMERIC_FEATURES, ORDINAL_FEATURES, RAW_FEATURES,
    TARGET_COLUMN, load_raw, split_features_target,
)
sns.set_theme(style='whitegrid')

train = load_raw('train')
test = load_raw('test')
X, y = split_features_target(train)
print('train', train.shape, '| test', test.shape)
train.head()

## Q1 — Target balance

In [ ]:
rate = y.mean()
print(f'positive rate = {rate:.4f}  ({y.sum():,} / {len(y):,})')
ax = y.value_counts(normalize=True).sort_index().plot.bar()
ax.set_title(f'Target balance (positive rate {rate:.3f})'); ax.set_xlabel(TARGET_COLUMN)
plt.tight_layout(); plt.savefig(FIG / 'target_balance.png', dpi=120); plt.show()

## Q2 — Missing values & dtypes

In [ ]:
na = train.isna().sum()
print('total missing:', int(na.sum()))
print(na[na > 0] if na.sum() else 'none')
train[list(RAW_FEATURES)].describe().T

## Q3 — Numeric distributions & skew

In [ ]:
cols = list(NUMERIC_FEATURES)
fig, axes = plt.subplots(1, len(cols), figsize=(5 * len(cols), 4))
for ax, c in zip(np.atleast_1d(axes), cols):
    sns.histplot(train, x=c, hue=TARGET_COLUMN, bins=40, stat='density',
                 common_norm=False, ax=ax)
    ax.set_title(f'{c} (skew {train[c].skew():.2f})')
plt.tight_layout(); plt.savefig(FIG / 'numeric_distributions.png', dpi=120); plt.show()

## Q4 — Cardinality of binary/ordinal features

In [ ]:
card = {c: train[c].nunique() for c in RAW_FEATURES}
pd.Series(card).sort_values().to_frame('n_unique')

## Q5 — Single-feature association with the target (univariate AUC)
Because the metric is ROC-AUC, rank each feature by the AUC it achieves alone
(using the sign that makes it positively associated).

In [ ]:
from sklearn.metrics import roc_auc_score
from src.preprocessing import prepare_tree_features

# Encode everything (ordinals -> codes, nominals -> one-hot) so each column is
# numeric, then rank features by the AUC they achieve alone.
Xe = prepare_tree_features(X)
aucs = {}
for c in Xe.columns:
    a = roc_auc_score(y, Xe[c])
    aucs[c] = max(a, 1 - a)
auc_s = pd.Series(aucs).sort_values(ascending=False)
ax = auc_s.head(25).plot.barh(figsize=(7, 9)); ax.invert_yaxis()
ax.set_title('Univariate ROC-AUC vs target (top 25)'); ax.set_xlabel('AUC')
plt.tight_layout(); plt.savefig(FIG / 'univariate_auc.png', dpi=120); plt.show()
auc_s.head(15)

## Q6 — Feature correlations

In [ ]:
num_cols = list(NUMERIC_FEATURES) + list(BINARY_FEATURES)
corr = train[num_cols + [TARGET_COLUMN]].corr(numeric_only=True)
plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap='coolwarm', center=0, square=True, cbar_kws={'shrink': .6})
plt.title('Correlation matrix (numeric + binary)'); plt.tight_layout()
plt.savefig(FIG / 'correlation_matrix.png', dpi=120); plt.show()
corr[TARGET_COLUMN].drop(TARGET_COLUMN).sort_values(key=np.abs, ascending=False).head(12)

## Q7 — Train vs test distribution similarity

In [ ]:
summary = pd.DataFrame({
    'train_mean': train[list(RAW_FEATURES)].mean(),
    'test_mean': test[list(RAW_FEATURES)].mean(),
})
summary['abs_diff'] = (summary['train_mean'] - summary['test_mean']).abs()
summary.sort_values('abs_diff', ascending=False)

## Q8 — Engineered feature sanity check
Confirm the engineered composites separate the classes.

In [ ]:
from src.features import build_features, ENGINEERED_FEATURES
from sklearn.metrics import roc_auc_score
eng = build_features(X)
eng_auc = {c: (lambda a: max(a, 1 - a))(roc_auc_score(y, eng[c])) for c in ENGINEERED_FEATURES}
pd.Series(eng_auc).sort_values(ascending=False).to_frame('univariate_auc')

## Summary
Record conclusions in `reports/EDA_FINDINGS.md`: positive rate (~0.62), missing
values, skewed numerics, strongest predictors (check the univariate-AUC ranking —
expected leaders include the lipid ratios, `bmi`, `age`, blood pressure, and the
history flags), correlation clusters (lipids; blood pressure), and train/test
similarity.